# recount2 CLAMP models with C2CP prior — Pathway Coverage vs Sample Size (fixed K multiplier)

**Environment:** `clamp-analyses`

Same pipeline as `01_recount2_coverage.ipynb` but instead of estimating `CLAMP_K`
via `num.pc()`, uses pre-computed values from
`data/archs4/subsampled_number_of_lvs.tsv`.

The TSV provides one row per (sample_size × seed) combination with columns:
- `number_of_latent_variables` → used directly as `CLAMP_K`
- `seed`
- `sample_size`

Sample sizes: 500, 1000, 2000, 4000, 8000, 16000, 32000 (5 seeds each)

Output: `output/recount2_multiplier/c2cp_subsample_{N}_seed_{seed}/`

## Load libraries

In [11]:
start_time <- Sys.time()
cat("recount2 CLAMP coverage multiplier analysis started at:", format(start_time), "\n")

recount2 CLAMP coverage multiplier analysis started at: 2026-02-24 22:29:01 


In [12]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)

source(here("config.R"))

## Configuration

In [13]:
output_data_dir <- file.path(config$GENERAL$OUTPUT_DIR, "recount2_multiplier")
dir.create(output_data_dir, showWarnings = FALSE, recursive = TRUE)

# C2CP prior path
data_path <- here("data", "archs4")

# Input files (same as 00_recount2.ipynb)
out_dir   <- here("data", "recount2")
rds_file  <- file.path(out_dir, "recount2_PLIER_data", "recount_data_prep_PLIER.RDS")
rpkm_file <- file.path(out_dir, "recount2_PLIER_data", "recount_rpkm.RDS")

# Load pre-computed CLAMP_K and seeds per (sample_size x seed) from TSV
lvs_tsv <- read.table(
    here("data", "archs4", "subsampled_number_of_lvs.tsv"),
    header     = TRUE,
    sep        = "\t",
    colClasses = c(number_of_latent_variables = "integer",
                   seed                       = "integer",
                   sample_size                = "integer")
)

# Sort by sample_size ascending (then seed) so runs execute 500 → 32000
lvs_tsv <- lvs_tsv[order(lvs_tsv$sample_size, lvs_tsv$seed), ]
rownames(lvs_tsv) <- NULL

sample_sizes <- sort(unique(lvs_tsv$sample_size))

block_size <- config$GENERAL$CHUNK_SIZE
N_CORES    <- config$recount2$N_CORES

message("Output dir   : ", output_data_dir)
message("Sample sizes : ", paste(sample_sizes, collapse = ", "))
message("Total runs   : ", nrow(lvs_tsv))
print(lvs_tsv)

Output dir   : /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2_multiplier

Sample sizes : 500, 1000, 2000, 4000, 8000, 16000, 32000

Total runs   : 35



   number_of_latent_variables seed sample_size
1                          30 2876         500
2                          27 4089         500
3                          30 7883         500
4                          53 8828         500
5                          22 9401         500
6                          48 2876        1000
7                          49 4089        1000
8                          38 7883        1000
9                          29 8828        1000
10                         49 9401        1000
11                         68 2876        2000
12                         91 4089        2000
13                         18 7883        2000
14                        108 8828        2000
15                         42 9401        2000
16                        221 2876        4000
17                        187 4089        4000
18                        105 7883        4000
19                        156 8828        4000
20                        204 9401        4000
21           

## Preprocess recount2 data

Identical to `00_recount2.ipynb`: Ensembl→HGNC mapping via biomaRt,
FBM creation, `preprocessCLAMPFBM`, `zscoreCLAMPFBM`.
Skipped on subsequent runs if the filtered FBM already exists.

In [14]:
# Read gene symbols and sample names from the PLIER-prepared RDS
data_prep <- readRDS(rds_file)

meta <- list()
meta$gene_symbols <- rownames(data_prep$rpkm.cm)
meta$samples      <- colnames(data_prep$rpkm.cm)

rm(data_prep)

In [15]:
# Ensembl → HGNC symbol mapping via biomaRt (same as 00_recount2.ipynb)
rpkm.df <- readRDS(rpkm_file)

mart <- biomaRt::useDataset("hsapiens_gene_ensembl",
                            biomaRt::useMart("ensembl"))

rpkm.df$ensembl_gene_id <- unlist(lapply(strsplit(rpkm.df$ENSG, "[.]"), `[[`, 1))

gene.df <- biomaRt::getBM(
    filters    = "ensembl_gene_id",
    attributes = c("ensembl_gene_id", "hgnc_symbol"),
    values     = rpkm.df$ensembl_gene_id,
    mart       = mart
)
gene.df <- gene.df %>% dplyr::filter(complete.cases(.))

rpkm.df <- dplyr::inner_join(gene.df, rpkm.df,
                             by = "ensembl_gene_id",
                             relationship = "many-to-many")

rownames(rpkm.df) <- make.names(rpkm.df$hgnc_symbol, unique = TRUE)
rpkm.df <- rpkm.df %>% dplyr::select(-c(ensembl_gene_id:ENSG))

data_mat <- rpkm.df[meta$gene_symbols, meta$samples]
rm(rpkm.df)

data_mat <- as.matrix(as.data.table(data_mat))

In [16]:
# Remove stale FBM files before recreating (same as 00_recount2.ipynb)
fbm_files <- c(
    file.path(output_data_dir, "FBMrecount2_cov.bk"),
    file.path(output_data_dir, "FBMrecount2_cov_preproc.bk"),
    file.path(output_data_dir, "FBMrecount2_cov_preproc_filtered.bk")
)
for (f in fbm_files) {
    if (file.exists(f)) {
        unlink(f)
        message("Removed stale FBM file: ", f)
    }
}

In [17]:
# Create the FBM
n_genes_raw <- length(meta$gene_symbols)
n_samps_raw <- length(meta$samples)

fbm_file    <- file.path(output_data_dir, "FBMrecount2_cov")
recount2FBM <- FBM(
    nrow        = n_genes_raw,
    ncol        = n_samps_raw,
    backingfile = fbm_file,
    create_bk   = TRUE
)

In [18]:
# Populate FBM in row-blocks
n_blocks <- ceiling(n_genes_raw / block_size)
for (i in seq_len(n_blocks)) {
    start_row <- (i - 1) * block_size + 1
    end_row   <- min(i * block_size, nrow(data_mat))
    recount2FBM[start_row:end_row, ] <- as.matrix(data_mat[start_row:end_row, ])
}
rm(data_mat)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,5915380,316.0,11005321,587.8,11005321,587.8
Vcells,10726936,81.9,3722980697,28404.1,4743617352,36191.0


In [19]:
# Preprocess and z-score FBM
prep_recount2 <- preprocessCLAMPFBM(
    fbm         = recount2FBM,
    mean_cutoff = config$recount2$GENES_MEAN_CUTOFF,
    var_cutoff  = config$recount2$GENES_VAR_CUTOFF
)

recount2_fbm_filt <- prep_recount2$fbm_filtered
recount2_rowStats <- prep_recount2$rowStats

Applying log2 transformation

Filling NAs with 0



In [20]:
zscoreCLAMPFBM(recount2_fbm_filt, recount2_rowStats)

Applying Z-score transformation



In [21]:
samples        <- meta$samples
recount2_genes <- meta$gene_symbols[prep_recount2$kept_rows]

n_genes       <- nrow(recount2_fbm_filt)
n_samps_total <- ncol(recount2_fbm_filt)

message("Preprocessed FBM: ", n_genes, " genes x ", n_samps_total, " samples")

# Save gene list so downstream notebooks can compute matched pathway count
saveRDS(recount2_genes, file.path(output_data_dir, "recount2_genes.rds"))
saveRDS(samples,        file.path(output_data_dir, "recount2_samples.rds"))

Preprocessed FBM: 6000 genes x 37032 samples



## Load C2CP pathway prior

In [22]:
# Load C2CP pathway
c2_gmt <- CLAMP:::read_gmt(file.path(data_path, "c2.cp.v2026.1.Hs.symbols.gmt"))
names(c2_gmt) <- paste0("C2CP_", names(c2_gmt))
c2_pathMat <- gmtListToSparseMat(list(C2CP = c2_gmt))
C2CP_matched <- getMatchedPathwayMat(c2_pathMat, recount2_genes)
message("Loaded and matched C2CP pathway matrix")

There are 5691 genes in the intersection between data and prior

Removing 1446 pathways

Loaded and matched C2CP pathway matrix



## Run CLAMP models — loop over sample sizes and seeds (CLAMP_K from TSV)

In [23]:
results_summary <- data.frame(
    sample_size = integer(),
    seed        = integer(),
    n_samples   = integer(),
    CLAMP_K     = integer(),
    n_lvs_total = integer(),
    n_lvs_auc70 = integer(),
    n_lvs_auc90 = integer(),
    stringsAsFactors = FALSE
)

message("Total samples in preprocessed FBM: ", n_samps_total)

for (i in seq_len(nrow(lvs_tsv))) {

    n_target     <- lvs_tsv$sample_size[i]
    current_seed <- lvs_tsv$seed[i]
    CLAMP_K      <- lvs_tsv$number_of_latent_variables[i]

    message("\n", strrep("=", 60))
    message("SAMPLE SIZE: ", n_target,
            " | SEED: ", current_seed,
            " | CLAMP_K: ", CLAMP_K)
    message(strrep("=", 60))

    output_dir <- file.path(
        output_data_dir,
        paste0("c2cp_subsample_", n_target, "_seed_", current_seed)
    )
    dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

    # Skip if already complete
    if (file.exists(file.path(output_dir, "CLAMPfull_C2CP.rds"))) {
        message("Output already exists — skipping.")
        next
    }

    # ── Sample selection ──────────────────────────────────────────────
    set.seed(current_seed)
    n_draw     <- min(n_target, n_samps_total)
    sample_idx <- sort(sample(seq_len(n_samps_total), n_draw))
    n_samples  <- length(sample_idx)
    samp_names <- samples[sample_idx]

    message("Selected ", n_samples, " samples")

    saveRDS(
        list(
            sample_size     = n_target,
            seed            = current_seed,
            CLAMP_K         = CLAMP_K,
            n_samples       = n_samples,
            sample_idx      = sample_idx,
            sample_names    = samp_names,
            sampling_method = "random_sampling"
        ),
        file = file.path(output_dir, "subsample_info.rds")
    )

    # ── Subsampled FBM ────────────────────────────────────────────────
    message("Creating subsampled FBM...")
    fbm_sub_file <- file.path(output_dir, "fbm_subsampled")
    Y_sub <- big_copy(
        recount2_fbm_filt,
        ind.col     = sample_idx,
        backingfile = fbm_sub_file
    )

    # ── SVD ───────────────────────────────────────────────────────────
    message("Computing SVD...")
    SVD_K <- min(n_samples - 1, n_genes - 1)

    if (N_CORES > 1) {
        options(bigstatsr.check.parallel.blas = FALSE)
        blas_nproc <- getOption("default.nproc.blas")
        options(default.nproc.blas = NULL)
    }

    svd_result <- big_randomSVD(Y_sub, k = SVD_K, ncores = N_CORES)

    if (N_CORES > 1) {
        options(bigstatsr.check.parallel.blas = TRUE)
        options(default.nproc.blas = blas_nproc)
    }

    valid_idx    <- which(!is.nan(svd_result$d))
    svd_result$d <- svd_result$d[valid_idx]
    svd_result$u <- svd_result$u[, valid_idx, drop = FALSE]
    svd_result$v <- svd_result$v[, valid_idx, drop = FALSE]

    saveRDS(svd_result, file.path(output_dir, "svd.rds"))
    saveRDS(CLAMP_K,    file.path(output_dir, "CLAMP_K.rds"))
    message("CLAMP K = ", CLAMP_K, " (from TSV)")

    # ── CLAMPbase ─────────────────────────────────────────────────────
    message("Running CLAMPbase...")
    baseRes <- CLAMPbase(
        Y       = Y_sub,
        svdres  = svd_result,
        trace   = TRUE,
        clamp_k = CLAMP_K
    )

    baseRes$Z <- data.frame(baseRes$Z)
    rownames(baseRes$Z) <- recount2_genes
    baseRes$B <- data.frame(baseRes$B)
    colnames(baseRes$B) <- samp_names

    saveRDS(baseRes, file.path(output_dir, "CLAMPbase.rds"))

    model_dir <- file.path(output_dir, "CLAMPbase")
    dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
    write.csv(baseRes$B, file.path(model_dir, "B.csv"))
    write.csv(baseRes$Z, file.path(model_dir, "Z.csv"))

    # ── CLAMPfull with C2CP prior ─────────────────────────────────────
    message("Running CLAMPfull with C2CP prior...")
    fullRes <- CLAMPfull(
        Y                 = Y_sub,
        svdres            = svd_result,
        priorMat          = C2CP_matched,
        clamp.base.result = baseRes,
        use_cpp           = TRUE,
        trace             = TRUE,
        clamp_k           = CLAMP_K
    )

    fullRes$Z <- data.frame(fullRes$Z)
    rownames(fullRes$Z) <- recount2_genes
    fullRes$B <- data.frame(fullRes$B)
    colnames(fullRes$B) <- samp_names
    fullRes$summary <- fullRes$summary %>%
        dplyr::rename(LV = LV_index) %>%
        dplyr::mutate(LV = paste0("LV", LV))

    saveRDS(fullRes, file.path(output_dir, "CLAMPfull_C2CP.rds"))

    model_dir <- file.path(output_dir, "CLAMPfull_C2CP")
    dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
    write.csv(fullRes$B,       file.path(model_dir, "B.csv"))
    write.csv(fullRes$Z,       file.path(model_dir, "Z.csv"))
    write.csv(fullRes$summary, file.path(model_dir, "summary.csv"))

    # ── Record summary ────────────────────────────────────────────────
    n_lvs_total <- nrow(fullRes$summary)
    n_lvs_auc70 <- sum(fullRes$summary$AUC >= 0.70, na.rm = TRUE)
    n_lvs_auc90 <- sum(fullRes$summary$AUC >= 0.90, na.rm = TRUE)

    results_summary <- rbind(results_summary, data.frame(
        sample_size = n_target,
        seed        = current_seed,
        n_samples   = n_samples,
        CLAMP_K     = CLAMP_K,
        n_lvs_total = n_lvs_total,
        n_lvs_auc70 = n_lvs_auc70,
        n_lvs_auc90 = n_lvs_auc90
    ))

    # ── Clean up memory and subsampled FBM ───────────────────────────
    rm(Y_sub, svd_result, baseRes, fullRes)
    gc()
}

message("\nAll runs completed!")

Total samples in preprocessed FBM: 37032



SAMPLE SIZE: 500 | SEED: 2876 | CLAMP_K: 30


Selected 500 samples

Creating subsampled FBM...

Computing SVD...

CLAMP K = 30 (from TSV)

Running CLAMPbase...

****

CLAMP k is set to 30

L1 is set to 34.4000638911045

L2 is set to 103.200191673313

Progress 1 / 200 | Bdiff=0.520206, minCor=0.901199

Progress 2 / 200 | Bdiff=0.028094, minCor=0.967881

Progress 3 / 200 | Bdiff=0.009434, minCor=0.992561

Progress 4 / 200 | Bdiff=0.005134, minCor=0.994312

Progress 5 / 200 | Bdiff=0.003533, minCor=0.994890

Progress 6 / 200 | Bdiff=0.002781, minCor=0.995336

Progress 7 / 200 | Bdiff=0.002335, minCor=0.995763

Progress 8 / 200 | Bdiff=0.002034, minCor=0.996117

Progress 9 / 200 | Bdiff=0.001811, minCor=0.996377

Progress 10 / 200 | Bdiff=0.001641, minCor=0.996618

Progress 11 / 200 | Bdiff=0.001511, minCor=0.996887

Progress 12 / 200 | Bdiff=0.001403, minCor=0.997242

Progress 13 / 200 | Bdiff=0.001309, minCor=0.997651

Progress 14 / 200 | Bdiff

## Save and display results summary

In [ ]:
saveRDS(
    results_summary,
    file.path(output_data_dir, "subsample_results_summary_multiplier.rds")
)
write.csv(
    results_summary,
    file.path(output_data_dir, "subsample_results_summary_multiplier.csv"),
    row.names = FALSE
)

print(results_summary)

In [ ]:
end_time     <- Sys.time()
elapsed_time <- end_time - start_time
cat("Analysis completed at:", format(end_time), "\n")
cat("Total elapsed time   :", format(elapsed_time), "\n")